# BC카드 공모전 — 2단계 EDA (전체 추이 · 히트맵 · baseline · H2/H3 검증)

이 노트북은 1단계(`BC카드_1단계_전처리.ipynb`)의 결과물인 `bc_preprocessed.pkl`, `bc_stage3_signals.pkl`을 그대로 불러와서 이어서 작업한다 — 1단계 노트북을 먼저 위에서 끝까지 실행해서 두 pkl 파일이 구글 드라이브 `BC카드` 폴더에 저장되어 있어야 한다.

다루는 내용:
1. 외국인 세그먼트 전체 월별 추이 (계절성 육안 확인)
2. 시도별 × 업종별 소비 히트맵
3. 기존 유명 관광지 baseline 확립
4. H2(사각지대) 산점도 검증
5. H3(지속성) 홀드아웃 검증

## 0. 환경 설정 & 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/BC카드'

with open(f'{BASE_DIR}/bc_preprocessed.pkl', 'rb') as f:
    preprocessed = pickle.load(f)

with open(f'{BASE_DIR}/bc_stage3_signals.pkl', 'rb') as f:
    stage3 = pickle.load(f)

months = preprocessed['months']
pivot_amt = preprocessed['pivot_amt']
pivot_cnt = preprocessed['pivot_cnt']
national_amt = preprocessed['national_amt']
national_cnt = preprocessed['national_cnt']
pivot_amt_reliable = preprocessed['pivot_amt_reliable']
pivot_cnt_reliable = preprocessed['pivot_cnt_reliable']

cnt_scores = stage3['cnt_scores']
cnt_scores_clean = stage3['cnt_scores_clean']
excluded_artifacts = stage3['excluded_artifacts']
region_summary = stage3['region_summary']

print("months:", months)
print("pivot_cnt_reliable shape:", pivot_cnt_reliable.shape)
print("cnt_scores 개수:", len(cnt_scores))


---
## 1. 외국인 세그먼트 전체 월별 추이

계절성이 어느 정도 있는지 육안으로 먼저 확인한다 — 이게 3단계에서 "전국 평균 성장률 대비 상대 초과성장"으로 보정하는 이유다(모든 지역이 겪는 공통 등락은 상쇄해야 함).

In [ ]:
monthly_total_cnt = national_cnt.sum(axis=0)
monthly_total_amt = national_amt.sum(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(months, monthly_total_cnt.values, marker='o', color='#e74c3c')
axes[0].set_title('외국인 세그먼트 전체 월별 CNT 추이')
axes[0].set_xlabel('기준년월')
axes[0].set_ylabel('CNT 합계')

axes[1].plot(months, monthly_total_amt.values, marker='o', color='#3498db')
axes[1].set_title('외국인 세그먼트 전체 월별 AMT 추이')
axes[1].set_xlabel('기준년월')
axes[1].set_ylabel('AMT 합계 (원)')

plt.tight_layout()
plt.savefig('overall_trend.png', dpi=150)
plt.show()

print(monthly_total_cnt)


---
## 2. 시도별 × 업종별 소비 히트맵

17개 시도 × 11개 업종 조합의 6개월 누적 CNT를 히트맵으로 확인한다. (원본 `foreign` 데이터프레임 없이도 `pivot_cnt`를 시도 단위로 재집계하면 동일하게 만들 수 있다.)

In [ ]:
# 한글 폰트 설정 (Colab 기본 환경엔 한글 폰트가 없어 별도 설치 필요)
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

sido_buz = pivot_cnt.groupby(level=['SIDO_NM', 'TP_BUZ_NM_CLEAN']).sum().sum(axis=1).unstack('TP_BUZ_NM_CLEAN').fillna(0)
sido_buz = sido_buz.loc[sido_buz.sum(axis=1).sort_values(ascending=False).index]  # 총합 큰 시도 순 정렬

fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(sido_buz.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(sido_buz.columns)))
ax.set_xticklabels(sido_buz.columns, rotation=45, ha='right')
ax.set_yticks(range(len(sido_buz.index)))
ax.set_yticklabels(sido_buz.index)
ax.set_title('시도별 × 업종별 외국인 소비(CNT) 히트맵 (6개월 누적)')
fig.colorbar(im, ax=ax, label='CNT 합계')
plt.tight_layout()
plt.savefig('sido_buz_heatmap.png', dpi=150)
plt.show()

print(sido_buz)


**육안 확인**: 경기·서울·제주 등 인구·관광객 규모가 큰 시도가 진하게 나오는 게 정상 — 이게 바로 아래에서 만드는 "이미 큰 곳" baseline과 같은 그림이다. 실제 후보 발굴은 시군구×업종 단위(3단계)에서 상대 초과성장률로 이미 진행했다.

---
## 3. 기존 관광지 baseline 확립

필터링 전 전체 `pivot_cnt`를 시군구 단위로 묶어 6개월 누적 CNT(외국인 세그먼트 절대 소비 규모)를 계산한다.

In [ ]:
region_total = pivot_cnt.groupby(level=['SIDO_NM', 'CCG_NM']).sum().sum(axis=1)
region_total_sorted = region_total.sort_values(ascending=False)

print("절대 소비규모(6개월 누적 CNT) 상위 15개 지역:")
print(region_total_sorted.head(15))


### 예상 밖의 발견 — "외국인" 세그먼트에 관광객만 있는 게 아니다

상위 15개를 실제로 보면 서울 중구·부산 해운대구 같은 전통 관광지가 아니라 **안산시 단원구·시흥시·화성시 만세구·평택시·광산구·아산시·김포시·김해시·경산시·달서구** 같은 산업단지 밀집지역이 대부분이다. 실제 유명 관광지들의 순위를 직접 찍어서 확인해본다.

In [ ]:
rank_df = region_total_sorted.reset_index()
rank_df.columns = ['SIDO_NM', 'CCG_NM', 'total_cnt']
rank_df['순위'] = rank_df.index + 1

famous_spots = [
    ('서울특별시', '중구'), ('서울특별시', '종로구'), ('서울특별시', '강남구'),
    ('부산광역시', '해운대구'), ('부산광역시', '중구'),
    ('인천광역시', '중구'), ('강원특별자치도', '강릉시'),
    ('경상북도', '경주시'), ('강원특별자치도', '속초시'),
]
for sido, ccg in famous_spots:
    row = rank_df[(rank_df['SIDO_NM'] == sido) & (rank_df['CCG_NM'] == ccg)]
    if len(row):
        r = row.iloc[0]
        print(f"  {sido} {ccg}: 순위 {r['순위']}위 / 총CNT {int(r['total_cnt'])}")
    else:
        print(f"  {sido} {ccg}: 최소건수 필터 등으로 데이터 없음")


**해석**: 서울 중구(27위)·종로구(44위)·해운대구(60위)·속초시(150위) 등 실제 관광지는 오히려 중위권이다. → `GENDER_CD=3`("외국인") 세그먼트는 관광객뿐 아니라 산업단지의 외국인 근로자 일상소비(편의점·슈퍼마켓)까지 섞여 있다는 뜻. 따라서 H2(사각지대) 검증의 "이미 알려진 곳" 비교군은 절대 소비규모 상위가 아니라 **실제로 유명한 관광지**만 따로 선정해야 한다.

---
## 4. H2 (사각지대) 산점도 검증

지역을 4개 그룹으로 나눈다: ① 3단계에서 확정한 후보군, ② 실제 유명 관광지, ③ 산업단지형 대량소비 지역, ④ 그 외. baseline(소비규모, x축)과 평균 상대초과성장률(y축)의 산점도를 그려 후보군이 "소비규모 하위·성장률 상위" 사분면에 속하는지 확인한다.

In [ ]:
sdf2 = cnt_scores.reset_index()
sdf2.columns = ['SIDO_NM', 'CCG_NM', 'TP_BUZ_NM_CLEAN', 'score']
region_growth = sdf2.groupby(['SIDO_NM', 'CCG_NM'])['score'].mean()

plot_df = pd.DataFrame({'baseline_size': region_total, 'avg_growth': region_growth}).dropna()

candidates = [('경상북도', '청송군'), ('경상북도', '봉화군'), ('강원특별자치도', '고성군'),
              ('강원특별자치도', '영월군'), ('경상북도', '영양군'), ('경상북도', '의성군'),
              ('강원특별자치도', '양양군'), ('충청남도', '보령시'), ('인천광역시', '옹진군')]
real_tourist = [('서울특별시', '중구'), ('서울특별시', '종로구'), ('서울특별시', '강남구'),
                 ('부산광역시', '해운대구'), ('인천광역시', '중구'), ('경상북도', '경주시'),
                 ('강원특별자치도', '속초시'), ('강원특별자치도', '강릉시'),
                 ('제주특별자치도', '제주시'), ('제주특별자치도', '서귀포시')]
industrial = [('경기도', '안산시 단원구'), ('경기도', '시흥시'), ('경기도', '화성시 만세구'),
              ('경기도', '평택시'), ('광주광역시', '광산구'), ('충청남도', '아산시'),
              ('경기도', '김포시'), ('경상남도', '김해시'), ('경상북도', '경산시'), ('대구광역시', '달서구')]


def grp(idx):
    if idx in candidates:
        return '오늘의 후보군'
    if idx in real_tourist:
        return '실제 유명 관광지'
    if idx in industrial:
        return '산업단지형 대량소비'
    return '기타'


plot_df['group'] = [grp(i) for i in plot_df.index]

print("그룹별 baseline·평균초과성장률 요약:")
print(plot_df.groupby('group')[['baseline_size', 'avg_growth']].agg(['mean', 'count']))


In [ ]:
colors = {'오늘의 후보군': '#e74c3c', '실제 유명 관광지': '#3498db',
          '산업단지형 대량소비': '#95a5a6', '기타': '#dddddd'}
order = ['기타', '산업단지형 대량소비', '실제 유명 관광지', '오늘의 후보군']

fig, ax = plt.subplots(figsize=(10, 7))
for g in order:
    sub = plot_df[plot_df['group'] == g]
    ax.scatter(sub['baseline_size'], sub['avg_growth'] * 100, label=g, c=colors[g],
               s=90 if g == '오늘의 후보군' else 45, alpha=0.9 if g == '오늘의 후보군' else 0.6,
               edgecolors='black' if g == '오늘의 후보군' else 'none', linewidths=0.8,
               zorder=3 if g == '오늘의 후보군' else 2)

for idx in candidates:
    if idx in plot_df.index:
        row = plot_df.loc[idx]
        ax.annotate(idx[1], (row['baseline_size'], row['avg_growth'] * 100),
                    xytext=(5, 5), textcoords='offset points', fontsize=9, color='#c0392b')

ax.set_xscale('log')
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xlabel('기존 소비 규모 (6개월 누적 CNT, log scale)')
ax.set_ylabel('평균 상대 초과성장률 (%p)')
ax.set_title('H2 검증: 소비규모(baseline) vs 초과성장률 — 사각지대는 좌상단')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('h2_scatter.png', dpi=150)
plt.show()


**결과**: 실제 유명 관광지(평균 baseline 115만, 평균초과성장 0.9%p)와 산업단지형 대량소비(평균 128만, 0.7%p)는 둘 다 이미 규모가 커서 성장이 정체된 반면, 오늘의 후보군(평균 baseline 3.4만, 평균초과성장 19.7%p)은 baseline이 1/30 수준인데 성장률은 20~30배 높다. → **H2 채택**: "소비규모 하위·성장률 상위" 사분면에 명확히 몰려 있음.

---
## 5. H3 (지속성) 홀드아웃 검증

6개월이라는 짧은 시계열의 한계를 보완하기 위해, 1~4월 데이터만으로 점수를 다시 매겼을 때도 후보로 뽑혔을지, 그리고 그 성장이 5~6월 실측 데이터에서도 이어지는지 홀드아웃 방식으로 확인한다. (아래 두 함수는 1단계 노트북 부록에서 정의했던 것과 동일 — pkl에는 함수가 저장되지 않으므로 여기서 다시 정의한다.)

In [ ]:
def relative_growth_score(pivot_sub, national_full, months, n_recent=2):
    growth_rows = []
    idx_pairs = list(zip(months[:-1], months[1:]))[-n_recent:]
    for (m0, m1) in idx_pairs:
        reg_g = pivot_sub[m1] / pivot_sub[m0].replace(0, np.nan) - 1
        buz_level = pivot_sub.index.get_level_values('TP_BUZ_NM_CLEAN')
        nat_g = (national_full[m1] / national_full[m0].replace(0, np.nan) - 1)
        nat_g_aligned = buz_level.map(nat_g)
        growth_rows.append(reg_g.values - nat_g_aligned.values)
    arr = np.vstack(growth_rows)
    score = np.nanmean(arr, axis=0)
    return pd.Series(score, index=pivot_sub.index).sort_values(ascending=False)


def excess_growth_excluding(row, nat_row, from_idx, to_idx):
    g_region = row.iloc[to_idx] / row.iloc[from_idx] - 1
    g_nat = nat_row.iloc[to_idx] / nat_row.iloc[from_idx] - 1
    return g_region - g_nat


In [ ]:
months_train = months[:4]  # 1~4월만 사용
train_scores = relative_growth_score(pivot_cnt_reliable, national_cnt, months_train, n_recent=2)

good_items = [
    ('경상북도', '청송군', '편의점'), ('경상북도', '청송군', '서양음식'), ('경상북도', '청송군', '슈퍼마켓'),
    ('경상북도', '봉화군', '편의점'), ('경상북도', '봉화군', '슈퍼마켓'), ('경상북도', '봉화군', '서양음식'), ('경상북도', '봉화군', '일반한식'),
    ('강원특별자치도', '고성군', '서양음식'), ('강원특별자치도', '고성군', '편의점'), ('강원특별자치도', '고성군', '슈퍼마켓'),
    ('강원특별자치도', '영월군', '편의점'), ('강원특별자치도', '영월군', '일반한식'), ('강원특별자치도', '영월군', '슈퍼마켓'),
    ('경상북도', '영양군', '슈퍼마켓'), ('경상북도', '영양군', '편의점'),
    ('경상북도', '의성군', '서양음식'), ('경상북도', '의성군', '슈퍼마켓'),
    ('강원특별자치도', '양양군', '서양음식'), ('강원특별자치도', '양양군', '편의점'), ('강원특별자치도', '양양군', '일반한식'),
]

print(f"{'지역·업종':<20} {'1~4월기반 점수':>14} {'4월->6월 실측 초과성장':>18}  지속여부")
persisted, total = 0, 0
for key in good_items:
    if key not in train_scores.index:
        continue
    t_score = train_scores.loc[key]
    row = pivot_cnt_reliable.loc[key]
    nat = national_cnt.loc[key[2]]
    actual = excess_growth_excluding(row, nat, 3, 5)  # 4월->6월 실측
    ok = actual > 0
    persisted += int(ok)
    total += 1
    label = f"{key[1]} {key[2]}"
    print(f"{label:<20} {t_score*100:>12.1f}%p {actual*100:>16.1f}%p   {'지속' if ok else '반전'}")

print(f"\n지속 비율: {persisted}/{total} ({persisted/total*100:.0f}%)")


**결과**: 20개 항목 중 20개(100%) 모두 4월→6월 실측 초과성장이 양수로 지속됨. 단, 청송군 편의점·서양음식, 의성군 서양음식, 양양군 편의점은 1~4월 데이터만으로는 오히려 음수 점수라 애초에 후보에서 탈락했을 항목들 — 5~6월에 급성장한 케이스라는 뜻. → **H3(지속성)은 강하게 확인**되지만, 현재 방식은 "이미 뜬 후의 확인"에 가깝고 조기 포착(선행성, H1)은 상대적으로 약할 수 있다는 한계를 6단계에 명시한다.

---
## 6. 2단계 EDA 결과 저장

In [ ]:
stage2_eda_results = {
    'region_total': region_total,
    'plot_df': plot_df,
    'h3_holdout': {
        'good_items': good_items,
        'persisted': persisted,
        'total': total,
    },
}

with open(f'{BASE_DIR}/bc_stage2_eda.pkl', 'wb') as f:
    pickle.dump(stage2_eda_results, f)
print("2단계 EDA 결과 저장 완료:", f'{BASE_DIR}/bc_stage2_eda.pkl')
